# 数据统计

In [1]:
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt
import math

from collections import defaultdict

In [2]:
import sys
sys.path.append('..')

## 数据处理

In [3]:
def id_process(df, is_ost=False):
    # 曲目，专辑添加唯一id
    df = df.copy()
    df['song_id_unique'] = 'song' + df['song_id'].astype(str)
    # df_album_fixed = df[['album_id', 'album_fixed']].drop_duplicates(subset=['album_fixed'], keep='first').reset_index(drop=True)
    # df_album_fixed['album_id_unique'] = 'album' + df_album_fixed['album_id'].astype(int).astype(str)
    # df_album_fixed = df_album_fixed[['album_id_unique', 'album_fixed']]
    df['song_year'] = df['publish_date'].str.split('-').str[0]
    if is_ost:
        df['album_fixed'] = df['song_year'] + '年'
        df['album_id_unique'] = 'album' + df['song_year']
        df['legend_type'] = df['song_year'] + '年'
    else:
        df['album_fixed'] = df['album_name']
        df['album_id_unique'] = 'album' + df['album_id'].astype(str)
        df['legend_type'] = df['album_name']
    # df['legend_order'] = df['legend_type']
    # 词唯一id
    df_words = df[['word', 'pos']].drop_duplicates().reset_index(drop=False)
    df_words['word_id'] = 'word' + df_words['index'].astype(str)
    df_words = df_words.drop('index', axis=1).reset_index(drop=True)
    df = df.merge(df_words, on=['word', 'pos'], how='left')
    return df

# 分词数据
名词： n, w
动词： v
形容词： v

In [4]:
def word_count_by_pos(df, pos, words_num=30, is_starts_with=True):
    if is_starts_with:
        df_sub = df[df['pos'].str.startswith(
            pos, na=False)]
        # 歌曲数
        word_cnt = df_sub.groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos'].str.startswith(
            pos, na=False)].groupby('word')['freq'].sum().reset_index()
    else:
        word_cnt = df[df['pos']==pos]['word'].groupby('word')['song_id'].nunique().reset_index().rename(columns={'song_id': 'song_num'})
        word_sum = df[df['pos']==pos].groupby('word')['freq'].sum().reset_index()
    if words_num:
        word_cnt = word_cnt.sort_values(by='song_num', ascending=False)
        res = word_cnt.head(words_num).merge(word_sum, on='word', how='left')
    else:
        res = word_cnt.merge(word_sum, on='word', how='left')
    res['order'] = 100 - res.index
    # res = res.rename(columns={
    #     'count': 'songs_num',
    # })
    return res

# 词图

## 二分图布局

In [5]:
# 120度弧线布局
def generate_symmetric_bipartite_layout(nodes):
    coords = {}
    
    # --- 1. 参数定义 ---
    # 定义两条对称弧线的几何参数
    arc_radius = 800
    arc_span = math.pi / 1.5  # 约 120 度
    # 计算弧线的垂直跨度 (用于决定 Word 长度)
    arc_vertical_span = 2 * arc_radius * math.sin(arc_span / 2)
    
    # --- 2. Word 节点 (画布中央直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧线跨度的 90%
    target_word_height = arc_vertical_span * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心向两端扩散逻辑: 0->0, 1->1, 2->-1, 3->2...
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        
        # Word 位于 x=0，且在 y 轴居中
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (两侧对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    # 将 Song 平均分为两组：左侧弧和右侧弧
    song_columns = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 弧线布局配置：[左侧弧, 右侧弧]
    # 左侧弧圆心在正 X，向左弯曲；右侧弧圆心在负 X，向右弯曲
    configs = [
        {"center_x": 0, "direction": -1}, # 右侧弧 (位于 Word 右侧)
        {"center_x": 0, "direction": 1}  # 左侧弧 (位于 Word 左侧)
    ]

    for col_idx, col_items in enumerate(song_columns):
        config = configs[col_idx]
        num_in_col = len(col_items)
        if num_in_col == 0: continue
        
        for row_idx, node in enumerate(col_items):
            # 从上到下均匀分布角度
            if num_in_col > 1:
                angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span
            else:
                angle = 0
            
            # 计算坐标
            # cos(angle) 决定 X 偏移，direction 决定是在圆心左侧还是右侧
            x = config["center_x"] + config["direction"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


In [6]:
# 112度弧形布局
def generate_embracing_layout(nodes):
    coords = {}
    
    # --- 1. 几何参数设定 ---
    arc_radius = 800           # 半径
    arc_span = math.pi / 1.6   # 弧度张角 (约112度)
    # 弧开口端点距离中心直线的水平间距
    horizontal_gap = 300       
    
    # 计算弧线端点的 Y 轴跨度 (用于对齐 Word)
    # y = r * sin(theta)
    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height
    
    # --- 2. Word 节点 (居中直线) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'], key=lambda x: x['degree'], reverse=True)
    num_words = len(word_nodes)
    
    # 长度为弧垂直跨度的 90%
    target_word_height = arc_total_height * 0.9
    word_y_gap = target_word_height / (num_words - 1) if num_words > 1 else 0

    for i, node in enumerate(word_nodes):
        # 中心扩散排序
        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1
        if i == 0: direction = 0
        coords[node['id']] = (0, round(rank * direction * word_y_gap, 2))

    # --- 3. Song 节点 (开口向内的对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]
    
    # 配置说明：
    # 为了让开口面向直线 (x=0)：
    # 左侧弧的圆心要在右侧，x 坐标为 (horizontal_gap + radius * cos(half_span))
    # 右侧弧的圆心要在左侧，x 坐标为 -(horizontal_gap + radius * cos(half_span))
    
    # 计算圆心位置，使得弧的端点正好落在 horizontal_gap 线上
    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1, "name": "Right"}, # 右侧弧，圆心在左，向右弯
        {"c_x": 0, "dir": -1, "name": "Left"}   # 左侧弧，圆心在右，向左弯
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)
        
        for row_idx, node in enumerate(col_items):
            # 角度分布
            angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span if num_in_col > 1 else 0
            
            # 计算 X: 圆心 + 方向 * (半径 * cos(角度))
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))
            
    return coords


In [28]:
# 中心直线节点间隔非线性 112度弧线
def generate_embracing_layout_with_expansion(nodes, stretch_factor=0.85):
    """
    生成开口面向直线的弧形布局
    优化：中间直线节点采用非线性分布，撑开中心间距
    stretch_factor: 拉伸系数控制】
    # 使用小于 1 的指数（如 0.5 是开根号）。
    # 指数越小，中心第一个节点与第二个节点的距离就越大，两端越拥挤。
    # 推荐值：0.5 - 0.7
    """
    coords = {}

    # --- 1. 几何参数设定 ---
    arc_radius = 800  # 半径
    arc_span = math.pi / 1.6  # 弧度张角 (约112度)
    # 弧开口端点距离中心直线的水平间距
    horizontal_gap = 300

    # 计算弧线端点的 Y 轴跨度 (用于对齐 Word)
    # y = r * sin(theta)
    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height

    # --- 2. Word 节点 (修正：中心间隔大，两端间隔小) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'],
                        key=lambda x: x['degree'],
                        reverse=True)
    num_words = len(word_nodes)

    # 长度为弧垂直跨度的 90%
    target_word_height = arc_total_height * 0.9
    half_height = target_word_height / 2

    # 【拉伸系数控制】
    # 使用小于 1 的指数（如 0.5 是开根号）。
    # 指数越小，中心第一个节点与第二个节点的距离就越大，两端越拥挤。
    # 推荐值：0.5 - 0.7
    # stretch_factor = 0.9

    max_rank = num_words // 2

    for i, node in enumerate(word_nodes):
        if i == 0:
            coords[node['id']] = (0, 0)
            continue

        rank = (i + 1) // 2
        direction = 1 if i % 2 != 0 else -1

        # 核心逻辑：(rank / max_rank) 的 p 次方
        # 当 p < 1 时，函数图像在原点附近坡度极陡，能有效撑开中心节点
        normalized_y = (rank / max_rank)**stretch_factor * half_height

        coords[node['id']] = (0, round(direction * normalized_y, 2))

    # --- 3. Song 节点 (开口向内的对称弧线) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]

    # 配置说明：
    # 为了让开口面向直线 (x=0)：
    # 左侧弧的圆心要在右侧，x 坐标为 (horizontal_gap + radius * cos(half_span))
    # 右侧弧的圆心要在左侧，x 坐标为 -(horizontal_gap + radius * cos(half_span))

    # 计算圆心位置，使得弧的端点正好落在 horizontal_gap 线上
    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {
            "c_x": 0,
            "dir": 1,
            "name": "Right"
        },  # 右侧弧，圆心在左，向右弯
        {
            "c_x": 0,
            "dir": -1,
            "name": "Left"
        }  # 左侧弧，圆心在右，向左弯
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)

        for row_idx, node in enumerate(col_items):
            # 角度分布
            angle = (arc_span / 2
                     ) - (row_idx /
                          (num_in_col - 1)) * arc_span if num_in_col > 1 else 0

            # 计算 X: 圆心 + 方向 * (半径 * cos(角度))
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))

    return coords

In [48]:
def generate_embracing_layout_with_expansion(nodes, stretch_factor=0.85):
    """
    生成开口面向直线的弧形布局
    优化：Word 节点按 degree 自上向下排列
    视觉：顶部（Degree大）间隔疏，向下（Degree小）间隔越密
    """
    coords = {}

    # --- 1. 几何参数设定 ---
    arc_radius = 800
    arc_span = math.pi / 1.6
    horizontal_gap = 300

    arc_half_height = arc_radius * math.sin(arc_span / 2)
    arc_total_height = 2 * arc_half_height

    # --- 2. Word 节点 (顶部疏，底部密) ---
    word_nodes = sorted([n for n in nodes if n['type'] == 'word'],
                        key=lambda x: x['degree'],
                        reverse=True)
    num_words = len(word_nodes)

    target_word_height = arc_total_height * 0.9
    
    # stretch_factor 说明：
    # 指数越小（如 0.4-0.6），顶部节点被推开的幅度越大，底部压缩越厉害。
    # 指数 = 1.0 时，为等间距分布。

    for i, node in enumerate(word_nodes):
        # 归一化进度 t: 从 0 (顶部) 到 1 (底部)
        t = i / (num_words - 1) if num_words > 1 else 0
        
        # 核心逻辑：利用幂函数特性映射 Y 坐标
        # 我们希望在 t 较小时 y 变化快，t 较大时 y 变化慢
        # 公式：y = (t^p) * 总高度 - half_height
        # 这样当 t=0 时 y=-half_height; 当 t=1 时 y=half_height
        y_val = (t ** stretch_factor) * target_word_height - (target_word_height / 2)

        coords[node['id']] = (0, round(y_val, 2))

    # --- 3. Song 节点 (保持不变) ---
    song_nodes = [n for n in nodes if n['type'] == 'song']
    mid_idx = len(song_nodes) // 2
    song_groups = [song_nodes[:mid_idx], song_nodes[mid_idx:]]

    edge_x_offset = arc_radius * math.cos(arc_span / 2)
    center_x_pos = horizontal_gap + edge_x_offset

    side_configs = [
        {"c_x": 0, "dir": 1},  # 右侧弧
        {"c_x": 0, "dir": -1}  # 左侧弧
    ]

    for col_idx, col_items in enumerate(song_groups):
        conf = side_configs[col_idx]
        num_in_col = len(col_items)

        for row_idx, node in enumerate(col_items):
            angle = (arc_span / 2) - (row_idx / (num_in_col - 1)) * arc_span if num_in_col > 1 else 0
            x = conf["c_x"] + conf["dir"] * (arc_radius * math.cos(angle))
            y = arc_radius * math.sin(angle)
            coords[node['id']] = (round(x, 2), round(y, 2))

    return coords

## 数据处理

In [7]:
def get_subset_data(df_raw, pos, words_num, is_starts_with, is_ost=False):
    # 1. 获取符合特定词性的高频词集合
    # 假设 word_count_by_pos 返回的是一个包含 'word' 列的 DataFrame
    words_set = word_count_by_pos(df_raw,
                                  pos=pos,
                                  words_num=words_num,
                                  is_starts_with=is_starts_with)
    target_words = set(words_set['word'])  # 转为 set 匹配速度更快

    # 2. 统一词性过滤逻辑
    if is_starts_with:
        mask = df_raw['pos'].str.startswith(pos, na=False)
    else:
        mask = df_raw['pos'] == pos

    # 3. 筛选、清洗并保留必要的列
    # 链式操作：过滤词性 -> 过滤高频词 -> 执行自定义清洗
    df_subset = df_raw[mask].copy()
    df_subset = df_subset[df_subset['word'].isin(target_words)]

    # 4. 统一词性标签（既然是 Subset，统一设为传入的 pos）
    df_subset['pos'] = pos
    # 统一词的id
    df_subset = id_process(df_subset, is_ost=is_ost)

    # 5. 生成年份排序映射（album_order）
    # 使用 factorize 可以直接一步生成按顺序排列的编码
    # 如果必须按年份数值排序，则先排序再 factorize
    unique_years = sorted(df_subset['song_year'].unique())
    year_to_order = {year: i for i, year in enumerate(unique_years)}
    df_subset['album_order'] = df_subset['song_year'].map(year_to_order)

    # 排序
    df_subset = df_subset.sort_values(by=['album_order'])
    return df_subset.reset_index(drop=True)

## json数据输出

In [49]:
def get_word_subset_graph(G_words_subset, df_subset, is_ost=False):
    word_graph_dict = defaultdict(list)
    words_degree = G_words_subset.degree
    nodes_subset = []
    for n in G_words_subset.nodes():
        n_dict = {
            'id': n,
            'degree': words_degree[n],
            'type': 'word' if 'word' in n else 'song',
        }
        nodes_subset.append(n_dict)
    # pos = generate_embracing_layout(nodes_subset)
    pos =  generate_embracing_layout_with_expansion(nodes_subset)
    for node in G_words_subset.nodes:
        nodes_dict = defaultdict(str)
        nodes_dict['id'] = node
        nodes_dict['size'] = words_degree[node]
        nodes_dict['x'] = pos[node][0]
        nodes_dict['y'] = pos[node][1]
        if 'word' in node:
            nodes_dict['label'] = df_subset[df_subset['word_id'] ==
                                            node]['word'].values[0]
            nodes_dict['node_type'] = 'word'
            nodes_dict['album_id'] = ""
            nodes_dict['album'] = ""
            nodes_dict['data'] = {'cluster': '词'}
        elif 'song' in node:
            nodes_dict['label'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['song_name_unique'].values[0]
            nodes_dict['node_type'] = 'song'
            nodes_dict['album_id'] = df_subset[
                df_subset['song_id_unique'] ==
                node]['album_id_unique'].values[0]
            nodes_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                            node]['album_fixed'].values[0]
            nodes_dict['album_order'] = int(df_subset[
                df_subset['song_id_unique'] == node]['album_order'].values[0])
            nodes_dict['tv_name'] = df_subset[
                df_subset['song_id_unique'] ==
                node]['tv_name'].values[0] if is_ost else ""
            nodes_dict['data'] = {
                'cluster':
                df_subset[df_subset['song_id_unique'] == node]
                ['album_fixed'].values[0]
            }
        word_graph_dict['nodes'].append(nodes_dict)
    for edge in G_words_subset.edges:
        edges_dict = defaultdict(str)
        edges_dict['source'] = edge[0]
        edges_dict['target'] = edge[1]
        for i in edge:
            if 'song' in i:
                edges_dict['album'] = df_subset[df_subset['song_id_unique'] ==
                                                i]['album_fixed'].values[0]
                edges_dict['album_order'] = int(df_subset[
                    df_subset['song_id_unique'] == i]['album_order'].values[0])
            else:
                edges_dict['album'] = ""
                edges_dict['album_order'] = ""

        word_graph_dict['edges'].append(edges_dict)
    return word_graph_dict

# main

In [9]:
pos_dict = {'n': '名词', 'a': '形容词', 'v': '动词'}

In [55]:
file_path_prefix = "data/jaychou/"
file_path_prefix = "data/mayday/"

In [56]:
df_words_raw = pd.read_csv(file_path_prefix + "cleared_words_data.csv")
df_words_raw

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,来,v,12,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,107709592,能,v,6,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
2,107709592,人生,n,6,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
3,107709592,有,v,4,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
4,107709592,是,v,4,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8387,519403016,天涯飞奔,id,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8388,519403016,回头,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8389,519403016,飞奔,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8390,519403016,请,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16


In [57]:
df_words = id_process(df_words_raw)

In [58]:
pos_type = "n"
df_subset = get_subset_data(df_words_raw, pos_type, words_num=50, is_starts_with=True)
df_subset

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,...,song_name_unique,album_id,publish_date,song_id_unique,song_year,album_fixed,album_id_unique,legend_type,word_id,album_order
0,4930732,风,n,2,0033rnKT3vTwVc,I Love You 无望,NaN,五月天,74,000Sp0Bz4JXH0o,...,I Love You 无望,96215,1999-07-07,song4930732,1999,第一张创作专辑,album96215,第一张创作专辑,word218,0
1,4930724,黑夜,n,2,001Sh6UI3dh9mE,拥抱,《想见你》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,...,拥抱,96215,1999-07-07,song4930724,1999,第一张创作专辑,album96215,第一张创作专辑,word1150,0
2,4930724,手,n,3,001Sh6UI3dh9mE,拥抱,《想见你》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,...,拥抱,96215,1999-07-07,song4930724,1999,第一张创作专辑,album96215,第一张创作专辑,word485,0
3,4930724,人,n,3,001Sh6UI3dh9mE,拥抱,《想见你》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,...,拥抱,96215,1999-07-07,song4930724,1999,第一张创作专辑,album96215,第一张创作专辑,word125,0
4,4930725,心,n,4,000whnSb2dWU4L,透露,NaN,五月天,74,000Sp0Bz4JXH0o,...,透露,96215,1999-07-07,song4930725,1999,第一张创作专辑,album96215,第一张创作专辑,word13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732,107726176,话,n,1,003DfSzw4PnmRb,人生有限公司,NaN,五月天,74,000Sp0Bz4JXH0o,...,人生有限公司,1393445,2016-07-21,song107726176,2016,自传,album1393445,自传,word1680,10
733,107726182,日子,n,1,0018vLuq1v6rCW,转眼,NaN,五月天,74,000Sp0Bz4JXH0o,...,转眼,1393445,2016-07-21,song107726182,2016,自传,album1393445,自传,word30,10
734,107726182,风景,n,1,0018vLuq1v6rCW,转眼,NaN,五月天,74,000Sp0Bz4JXH0o,...,转眼,1393445,2016-07-21,song107726182,2016,自传,album1393445,自传,word122,10
735,107726201,故事,n,1,0015O0nW0P0FCH,成名在望,NaN,五月天,74,000Sp0Bz4JXH0o,...,成名在望,1393445,2016-07-21,song107726201,2016,自传,album1393445,自传,word9,10


In [59]:
G_words = nx.from_pandas_edgelist(df_words, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_words_subset = nx.from_pandas_edgelist(df_subset, 'word_id', 'song_id_unique', edge_attr=True, create_using=nx.Graph())
G_songs = nx.from_pandas_edgelist(df_words, 'song_id_unique', 'album_id_unique', edge_attr=True, create_using=nx.Graph())
G_words.number_of_nodes(), G_words_subset.number_of_nodes(), G_songs.number_of_nodes()

(3514, 179, 142)

In [60]:
# 添加数据信息
singer = df_subset['artist_name'].values[0]
data_info = {
    'singer': singer,
    'title': f'{singer}录音室&精选辑',
    'pos': pos_dict[pos_type],
    'all_songs_num': df_words_raw['song_id'].nunique(),
    'pos_songs_num': df_subset['song_id'].nunique(),
}
data_info

{'singer': '五月天',
 'title': '五月天录音室&精选辑',
 'pos': '名词',
 'all_songs_num': 131,
 'pos_songs_num': 129}

In [61]:
word_graph_dict = get_word_subset_graph(G_words_subset, df_subset)
word_graph_dict['data_info'] = data_info
with open(file_path_prefix+'word_graph_data.json', 'w', encoding='utf-8') as f:
    json.dump(word_graph_dict, f, ensure_ascii=False, indent=4)